# Task 4.1a. Huấn luyện GCN và GGNN trên Graph A và Graph B

| | |
|---|---|
| **Mục đích** | So sánh biểu diễn AST (Graph A) với AST+CFG+DATA_DEP (Graph B) bằng GCN và GGNN, trên hai protocol |
| **Đầu vào** | Kaggle Dataset `DiverseVul 9077 - Processed Graph Data` (đầu ra của notebook `4_0_build_processed`) |
| **Đầu ra** | `results_4_1a_gnn/`: `results.csv`, `per_class.csv`, `confusion_matrices.json`, `training_history.json`, `summary.md` |
| **Máy chạy** | GPU (T4 hoặc P100), bật Internet |

Huấn luyện: class-weighted cross-entropy, chọn epoch tốt nhất theo Macro-F1 trên validation, 3 seed. Kết quả ghi dần nên phiên bị ngắt vẫn giữ được phần đã xong.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

GIT_BRANCH = "main"
REPO_URL = "https://github.com/meo225/software-bug-detection-using-graphs.git"
REPO_DIR = Path("/tmp/g41a/software-bug-detection-using-graphs")  # Ngoài thư mục output
if not Path("/kaggle/input").is_dir():
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle.")
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, REPO_URL, str(REPO_DIR)],
                   check=True, timeout=180, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
else:
    subprocess.run(["git", "pull", "origin", GIT_BRANCH], cwd=REPO_DIR, check=True, timeout=180)
print("Đã tải mã nguồn:", subprocess.run(["git", "log", "-1", "--oneline"], cwd=REPO_DIR, text=True, capture_output=True).stdout, flush=True)

Cloning into '/tmp/g41a/software-bug-detection-using-graphs'...


Đã tải mã nguồn: f7e0629 Merge pull request #11 from meo225/feature/4_1a_results



In [2]:
import os
def run(cmd):
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode:
        raise RuntimeError(f"Lệnh thất bại với mã {proc.returncode}")

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

torch 2.11.0+cu130 | CUDA: True Tesla T4


## 1. Xác định dữ liệu đầu vào

In [3]:
found = [p.parent for p in Path("/kaggle/input").rglob("metadata.json")
         if (p.parent / "project_wise" / "vocab.json").is_file()]
assert found, "Không thấy dữ liệu đã xử lý, hãy Add Input dataset DiverseVul 9077 - Processed Graph Data."
PROCESSED = found[0]
OUT_DIR = Path("/kaggle/working/results_4_1a_gnn")
print("processed:", PROCESSED)

processed: /kaggle/input/datasets/vtnmai/diversevul-9077-processed-graph-data/processed


## 2. Chạy thử nhanh

2 epoch, 200 mẫu mỗi split, chỉ để kiểm tra pipeline chạy được. Không dùng để đánh giá.

In [4]:
run([sys.executable, "scripts/train_gnn.py", "--processed-dir", str(PROCESSED),
     "--out-dir", "/tmp/g41a/smoke", "--protocols", "project_wise", "--seeds", "42",
     "--limit", "200", "--max-epochs", "2"])

Thiết bị: cuda; torch 2.11.0+cu130
project_wise/graph_a: train=200, validation=200, test=200; K=23
  [gcn seed=42] epoch 01 loss=3.0002 val_macro_f1=0.0197
  [gcn seed=42] epoch 02 loss=2.5879 val_macro_f1=0.0142
  [ggnn seed=42] epoch 01 loss=3.0036 val_macro_f1=0.0072
  [ggnn seed=42] epoch 02 loss=2.5169 val_macro_f1=0.0173
project_wise/graph_b: train=200, validation=200, test=200; K=23
  [gcn seed=42] epoch 01 loss=3.0014 val_macro_f1=0.0200
  [gcn seed=42] epoch 02 loss=2.5903 val_macro_f1=0.0142
  [ggnn seed=42] epoch 01 loss=2.9459 val_macro_f1=0.0072
  [ggnn seed=42] epoch 02 loss=2.3758 val_macro_f1=0.0231
# Kết quả GNN (Task 4.1a)

Trung bình ± độ lệch chuẩn theo seed.

| Protocol | Graph | Model | Split | Macro-F1 | Weighted-F1 | MCC | Accuracy | Tham số | Giây/seed |
| :--- | :--- | :--- | :--- | ---: | ---: | ---: | ---: | ---: | ---: |
| project_wise | graph_a | gcn | test | 0.0145 ± 0.0000 | 0.0667 ± 0.0000 | 0.0000 ± 0.0000 | 0.2000 ± 0.0000 | 3307799 | 4 |
| project_wi

## 3. Huấn luyện đầy đủ

2 graph × 2 model × 2 protocol × 3 seed.

In [5]:
run([sys.executable, "scripts/train_gnn.py", "--processed-dir", str(PROCESSED),
     "--out-dir", str(OUT_DIR), "--seeds", "42", "43", "44"])

Thiết bị: cuda; torch 2.11.0+cu130
project_wise/graph_a: train=6101, validation=762, test=936; K=23
  [gcn seed=42] epoch 01 loss=3.1380 val_macro_f1=0.0171
  [gcn seed=42] epoch 02 loss=3.0864 val_macro_f1=0.0089
  [gcn seed=42] epoch 03 loss=2.9014 val_macro_f1=0.0221
  [gcn seed=42] epoch 04 loss=2.6750 val_macro_f1=0.0200
  [gcn seed=42] epoch 05 loss=2.4737 val_macro_f1=0.0386
  [gcn seed=42] epoch 06 loss=2.2649 val_macro_f1=0.0476
  [gcn seed=42] epoch 07 loss=2.0892 val_macro_f1=0.0379
  [gcn seed=42] epoch 08 loss=1.9090 val_macro_f1=0.0443
  [gcn seed=42] epoch 09 loss=1.7407 val_macro_f1=0.0455
  [gcn seed=42] epoch 10 loss=1.5862 val_macro_f1=0.0426
  [gcn seed=42] epoch 11 loss=1.4738 val_macro_f1=0.0359
  [gcn seed=42] epoch 12 loss=1.3638 val_macro_f1=0.0446
  [gcn seed=42] epoch 13 loss=1.2360 val_macro_f1=0.0426
  [gcn seed=42] epoch 14 loss=1.1233 val_macro_f1=0.0409
  [gcn seed=43] epoch 01 loss=3.1378 val_macro_f1=0.0139
  [gcn seed=43] epoch 02 loss=3.1009 val_macr

## 4. Kết quả

Bảng trung bình ± độ lệch chuẩn theo seed. Tệp đầy đủ nằm trong `results_4_1a_gnn/`.

In [6]:
print((OUT_DIR / "summary.md").read_text(encoding="utf-8"))

# Kết quả GNN (Task 4.1a)

Trung bình ± độ lệch chuẩn theo seed.

| Protocol | Graph | Model | Split | Macro-F1 | Weighted-F1 | MCC | Accuracy | Tham số | Giây/seed |
| :--- | :--- | :--- | :--- | ---: | ---: | ---: | ---: | ---: | ---: |
| project_wise | graph_a | gcn | test | 0.0664 ± 0.0088 | 0.0897 ± 0.0331 | 0.0342 ± 0.0123 | 0.0905 ± 0.0253 | 3307799 | 91 |
| project_wise | graph_a | gcn | validation | 0.0493 ± 0.0012 | 0.0707 ± 0.0353 | 0.0259 ± 0.0058 | 0.0766 ± 0.0231 | 3307799 | 91 |
| project_wise | graph_a | ggnn | test | 0.0663 ± 0.0075 | 0.1072 ± 0.0126 | 0.0374 ± 0.0146 | 0.1129 ± 0.0131 | 3456407 | 116 |
| project_wise | graph_a | ggnn | validation | 0.0577 ± 0.0036 | 0.1218 ± 0.0382 | 0.0281 ± 0.0166 | 0.1133 ± 0.0354 | 3456407 | 116 |
| project_wise | graph_b | gcn | test | 0.0551 ± 0.0121 | 0.0830 ± 0.0129 | 0.0216 ± 0.0070 | 0.0823 ± 0.0129 | 3307799 | 131 |
| project_wise | graph_b | gcn | validation | 0.0545 ± 0.0024 | 0.0780 ± 0.0127 | 0.0173 ± 0.0025 | 0.0796 ± 